# Cite-Bench-TR · do AI models cite Turkish law correctly?

**The problem.** Lawyers, clients and expats now ask chatbots about Turkish law. The answers come with confident
citations: *"TBK m. 344"*, *"Article 54 of Law No. 6458"*. Some of those articles do not exist, were repealed, or belong
to a different law. Courts abroad already sanction lawyers for invented citations; nobody measures this for Turkish law.

**What you build here.** Two things, both small and fully offline:

1. **Katip Cite**, a checker (standard-library Python, 1.6 MB index) that finds every statute citation in a text and
   checks it against the official text of all 1,085 Turkish laws.
2. **Cite-Bench-TR**, 63 questions in 10 risk areas, asked in Turkish and English, that ranks language models by how
   often their answers are *safe*: no wrong citation, the right article, and the key fact (a deadline, a share).

**Time:** setup 2 min · checker 2 min · one local 4–8B model ~15–30 min on an M-series Mac · scoring 1 min.

**Pre-registered questions** (written before any model was run, 14 Nov 2026):

| | Question | How we answer it |
|---|---|---|
| Q1 | How often do models cite a statute article that is provably wrong? | wrong citations / all checkable citations, per model, with a bootstrap CI |
| Q2 | Are answers in **English** (expats, foreign clients) less safe than answers in Turkish? | paired TR vs EN on the same 63 questions, exact McNemar test, α = 0.05 |
| Q3 | Do models repeat **stale** rules (laws changed in 2023–24)? | regex for the old rule on the trap questions |
| Q4 | Does a bigger model buy safety? | safe-answer rate against parameter count, local models only |

Rules we keep: the question set is frozen before the first model run (changes go in `README.md` → *Amendments*), every
model gets the same prompt at temperature 0, and we report every model we run, including the bad results.

## 0 · Setup

Run from the repository root (the folder with `lab/`). Nothing here needs a GPU or an internet connection after step 1.

In [ ]:
import json, subprocess, sys, os
from pathlib import Path
ROOT = Path.cwd() if (Path.cwd() / "lab").exists() else Path.cwd().parents[1]
CITE = ROOT / "lab" / "cite"
sys.path.insert(0, str(CITE))
os.chdir(ROOT)
print("repo:", ROOT)

## 1 · The statute index (optional rebuild)

The index ships with the repo (`lab/cite/data/statutes.json.gz`). It comes from **Açık Mevzuat**
(github.com/onurcan-b/acik-mevzuat), which mirrors every law on mevzuat.adalet.gov.tr every day. Rebuild it when you
want a newer snapshot; the commit hash goes into every result, so a verdict can always be traced to a dated text.

✅ **Check:** about 1,085 laws and 38,000 articles; TBK has 649 articles, TMK 1,030.

In [ ]:
REBUILD = False
if REBUILD:
    subprocess.run(["git", "clone", "--depth", "1", "https://github.com/onurcan-b/acik-mevzuat", str(ROOT.parent / "acik-mevzuat")])
    subprocess.run([sys.executable, str(CITE / "build_index.py"), str(ROOT.parent / "acik-mevzuat")], check=True)
from katip_cite import Checker
ck = Checker()
L = ck.laws
print(ck.meta)
print(len(L), "laws;", "TBK", len(L["6098-turk-borclar-kanunu"]["arts"]["madde"]), "articles;", "TMK", len(L["4721-turk-medeni-kanunu"]["arts"]["madde"]))

## 2 · Try the checker

Paste any text: a petition, a chatbot answer, an English memo. Each citation gets one verdict:
`OK`, `NO_SUCH_ARTICLE`, `ARTICLE_REPEALED`, `LAW_REPEALED`, `NUMBER_NAME_CLASH`, `UNKNOWN_LAW` or `UNVERIFIED`.
Case-law citations are found and counted but not verified yet.

In [ ]:
text = """Kiracı temerrüde düşerse TBK m. 315 uygulanır; eski BK m. 260 da aynı yöndeydi. İş Kanunu md. 87 ve
TBK m. 712 de dikkate alınmalıdır. 6098 sayılı Türk Medeni Kanunu m. 2 dürüstlük kuralını düzenler.
Under Article 54 of the Law on Foreigners and International Protection, a foreigner may be deported."""
for c in ck.extract(text):
    print(f"{c.verdict:<18} {c.law_title or c.law_no} m. {c.article}  «{c.text}»  {c.note}")

**How good is the extractor?** `test_katip_cite.py` holds hand-written cases (Turkish and English forms, lists,
"aynı Kanunun", repealed codes, and sentences that must *not* be read as citations). It is our own set, so it shows the
checker does what we designed; step 6 measures it on real model answers.

✅ **Check:** every case passes.

In [ ]:
subprocess.run([sys.executable, str(CITE / "test_katip_cite.py")]);

## 3 · The questions, and a review sheet for lawyers

63 questions, 10 areas (labour, rent, family, inheritance, enforcement, criminal, procedure and deadlines, KVKK,
foreigners and expats, consumer and property). Each one lists the articles a good answer should cite (checked against
the law text) and, where there is one, the key fact as a pattern. They were drafted by Claude: **status `draft` until a
lawyer has reviewed them.**

The cell below writes `lab/cite/review/questions_review.csv`. Send it to a lawyer friend: one row per question, with
columns to tick or correct. Their changes go back into `make_questions.py` before the published run.

In [ ]:
import csv
qs = [json.loads(l) for l in (CITE / "data" / "questions.jsonl").read_text().splitlines()]
(CITE / "review").mkdir(exist_ok=True)
with open(CITE / "review" / "questions_review.csv", "w", newline="") as f:
    w = csv.writer(f)
    w.writerow(["id", "area", "risk", "question_tr", "question_en", "expected_articles", "key_fact_patterns", "trap",
                "OK? (evet/hayır)", "correct articles", "correct key fact", "comment"])
    for q in qs:
        w.writerow([q["id"], q["area"], q["risk"], q["q_tr"], q["q_en"], "; ".join(q["expect"]), " | ".join(q["must"]), q["trap"] or "", "", "", "", ""])
from collections import Counter
print(len(qs), "questions;", Counter(q["area"] for q in qs))
print("review sheet:", CITE / "review" / "questions_review.csv")

## 4 · Run models on your laptop

Any OpenAI-compatible server works. With **Ollama** (`brew install ollama`, then `ollama serve`):

```
ollama pull qwen3:8b        # or what you have: `ollama list`
ollama pull gemma3:4b
ollama pull llama3.2:3b
```

Answers are cached in `lab/cite/answers/<model>.jsonl`, so you can stop and resume. Start with a smoke test of 3
questions. For Qwen3 add `--no-think` (shorter, faster answers; the bench is about citations, not reasoning).

API models are optional and **not local**: they get only the public questions, never client data. Put the key in an
environment variable and pass its *name* (`--api-key-env OPENAI_API_KEY`), never the key itself.

In [ ]:
MODELS = ["qwen3:8b"]            # edit: the models you pulled
SMOKE = True                    # first True (3 questions), then False (all 63 x 2 languages)
for m in MODELS:
    args = [sys.executable, str(CITE / "bench.py"), "run", "--model", m] + (["--limit", "3"] if SMOKE else [])
    if m.startswith("qwen3"):
        args.append("--no-think")
    subprocess.run(args, check=True)

## 5 · Score and rank

`bench.py score` checks every answer and writes `results/<model>.json`; `make_leaderboard.py` writes
`LEADERBOARD.md` and `leaderboard.html`. Two reference rows show the scale: one quotes the expected article (the
ceiling, ~94%), one invents article numbers (the floor, 0%).

**How to read it:** *Safe answers* is the ranking. *Wrong citations %* answers Q1. *TR−EN* answers Q2 (a positive
number means Turkish answers are safer; p < 0.05 counts). *Stale rule* answers Q3.

In [ ]:
subprocess.run([sys.executable, str(CITE / "references.py")], check=True)
subprocess.run([sys.executable, str(CITE / "bench.py"), "score"], check=True)
subprocess.run([sys.executable, str(CITE / "make_leaderboard.py")], check=True)
print((CITE / "LEADERBOARD.md").read_text())

### Chart: safety against size (Q4)

Fill in the parameter counts of the models you ran (from `ollama show <model>`). Only local models are plotted.

In [ ]:
PARAMS_B = {"qwen3:8b": 8.2, "gemma3:4b": 4.3, "llama3.2:3b": 3.2}   # billions; edit
import matplotlib.pyplot as plt
res = [json.loads(p.read_text()) for p in (CITE / "results").glob("*.json") if not p.stem.startswith("ref-")]
pts = [(PARAMS_B[r["model"]], r["all"]["safe"], r["all"]["safe_ci"], r["name"]) for r in res if r["model"] in PARAMS_B and r["complete"]]
fig, ax = plt.subplots(figsize=(6.5, 4))
for x, y, ci, n in pts:
    ax.errorbar(x, 100 * y, yerr=[[100 * (y - ci[0])], [100 * (ci[1] - y)]], fmt="o", capsize=4)
    ax.annotate(n, (x, 100 * y), textcoords="offset points", xytext=(6, 4), fontsize=9)
ax.set_xscale("log"); ax.set_xlabel("parameters (billions, log)"); ax.set_ylabel("safe answers (%)"); ax.set_ylim(0, 100)
ax.set_title("Cite-Bench-TR: safety vs size (local models)"); ax.grid(alpha=.3)
fig.tight_layout(); fig.savefig(CITE / "results" / "safe_vs_size.png", dpi=160); plt.show()

## 6 · Audit the checker on real answers

Our tests are our own; here we measure on what models actually wrote. The cell samples 40 citations (stratified:
half flagged wrong, half OK) and writes `review/audit.csv`. Open each one, compare with the law text, and fill
`correct? (y/n)`. Then run the next cell: it reports the checker's precision on "wrong" flags (how often an accusation
is fair) and on "OK" verdicts. **Report both numbers next to the leaderboard.**

Also skim a few full answers for citations the checker *missed* (write them in `missed` with the answer id).

In [ ]:
import random
rnd = random.Random(14)
wrong, ok = [], []
for p in (CITE / "results").glob("*.json"):
    if p.stem.startswith("ref-"):
        continue
    r = json.loads(p.read_text())
    ans = {(a["id"], a["lang"]): a["answer"] for a in map(json.loads, (CITE / "answers" / f"{p.stem}.jsonl").read_text().splitlines()) if a.get("answer")}
    for (i, lang), text in ans.items():
        for c in ck.extract(text):
            if c.kind == "statute":
                (wrong if c.verdict in {"NO_SUCH_ARTICLE", "ARTICLE_REPEALED", "LAW_REPEALED", "NUMBER_NAME_CLASH", "UNKNOWN_LAW"} else ok).append((p.stem, i, lang, c))
sample = rnd.sample(wrong, min(20, len(wrong))) + rnd.sample(ok, min(20, len(ok)))
with open(CITE / "review" / "audit.csv", "w", newline="") as f:
    w = csv.writer(f)
    w.writerow(["model", "id", "lang", "citation", "law", "article", "verdict", "note", "article text (start)", "correct? (y/n)", "missed"])
    for m, i, lang, c in sample:
        w.writerow([m, i, lang, c.text, c.law_title or c.law_no, c.article, c.verdict, c.note, (c.article_text or "")[:200], "", ""])
print(len(wrong), "wrong-flagged and", len(ok), "OK citations; wrote", len(sample), "to review/audit.csv")

In [ ]:
rows = list(csv.DictReader(open(CITE / "review" / "audit.csv")))
done = [r for r in rows if r["correct? (y/n)"].strip().lower() in {"y", "n"}]
for label, keep in (("wrong flags", lambda r: r["verdict"] != "OK"), ("OK verdicts", lambda r: r["verdict"] == "OK")):
    xs = [r for r in done if keep(r)]
    if xs:
        good = sum(r["correct? (y/n)"].strip().lower() == "y" for r in xs)
        print(f"{label}: {good}/{len(xs)} correct ({100 * good / len(xs):.0f}%)")
print(len(done), "of", len(rows), "audited")

## 7 · Write the report

The cell writes `results/REPORT.md`: the leaderboard, the four answers, and the audit. Keep the wording plain:
*"Model X cited a provably wrong article in N of its answers (e.g. TBK m. 712, which does not exist)."*

In [ ]:
res = sorted([json.loads(p.read_text()) for p in (CITE / "results").glob("*.json") if not p.stem.startswith("ref-") and json.loads(p.read_text())["complete"]], key=lambda r: -r["all"]["safe"])
lines = ["# Cite-Bench-TR v0.1: results", "", f"Statute snapshot {ck.meta['commit'][:10]} ({ck.meta['commit_date']}). Questions: draft (lawyer review pending).", ""]
for r in res:
    a, g = r["all"], r["lang_gap"]
    lines.append(f"- **{r['name']}**: safe {100*a['safe']:.0f}% [{100*a['safe_ci'][0]:.0f}–{100*a['safe_ci'][1]:.0f}], "
                 f"wrong citations {a['wrong_citations']}/{a['citations']} ({100*a['wrong_rate']:.1f}%), "
                 f"TR {100*r['by_lang']['tr']['safe']:.0f}% vs EN {100*r['by_lang']['en']['safe']:.0f}% (McNemar p={g['mcnemar_p']:.3f}), stale {a['stale']}")
lines += ["", "Examples of wrong citations:", ""]
for r in res[:4]:
    for w in r["wrong_examples"][:3]:
        lines.append(f"- {r['name']}, {w['id']} ({w['lang']}): `{w['text']}` → {w['verdict']} ({w['note']})")
lines += ["", (CITE / "LEADERBOARD.md").read_text()]
(CITE / "results" / "REPORT.md").write_text("\n".join(lines))
print("\n".join(lines[:30]))